# Quickstart

This notebook walks the public API by example. Six primitives recur across every pipeline in the library, and the two examples below exercise them in their canonical configurations. A time lattice comes from `dbsp_time(n)`. A `Circuit` holds the operator graph. Each `Operator` joins that graph by calling its `connect` method. On top, an `Evaluator` mediates between user and circuit, exposing the `push` and `read` methods that drive everything below.

Two examples follow. The first pins down what each piece does on the simplest one-dimensional case imaginable. The second extends those same primitives into a doubly-incremental fixpoint whose feedback edge the caller must wire by hand, because no operator bakes such a cycle in. Both are tiny. Patterns exhibited within them scale directly to the larger pipelines that we will encounter later.

## Running sum

A single `Input` feeds a single `Integrate` on a one-dimensional lattice. Every value pushed into the input is accumulated inside the integrate. Reading the integrate at tick $k$ then returns the cumulative sum of all values pushed at or before $k$.

The recurrence is simple. At tick zero the integrate returns the value at tick zero. At every subsequent tick after that, it returns the previous integrate value plus the new input. Trivial. Yet the same recurrence underlies every more elaborate incremental computation built on top of the library, and that is what makes the running sum worth understanding before anything else.

In [1]:
from pydbsp.circuit import Circuit
from pydbsp.compute import ComputeCtx
from pydbsp.core import Antichain, dbsp_time
from pydbsp.evaluate import Evaluator
from pydbsp.operator import Input, Integrate
from pydbsp.progress import Time
from pydbsp.storage import DictStorage
from pydbsp.zset import ZSet, ZSetAddition

g = ZSetAddition[int]()
e = Evaluator[Time](
    circuit=Circuit[Time](),
    storage=DictStorage(),
    ctx=ComputeCtx(lattice=dbsp_time(1)),
    group=g,
)

src = Input[ZSet[int]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
cum = Integrate[ZSet[int]](group=g).connect(e.circuit, (src,))

e.push(src, ZSet({1: 1}))
e.push(src, ZSet({2: 1}))
e.push(src, ZSet({1: 1}))

[e.read(cum, (k,)).inner for k in range(3)]

[{1: 1}, {2: 1, 1: 1}, {1: 2, 2: 1}]

## Graph reachability

Reachability is doubly-incremental. Outer ticks deliver new edges. Inner ticks chase the fixpoint induced by those edges. The pipeline therefore lives on a two-dimensional lattice with `IndexedIncrementalReachabilityBody` serving as the body operator. Unlike the running sum above, the state-feedback cycle is not contained inside the operator.

Rather, the cycle is materialised externally. State lives in a two-dimensional `Input` whose values we push back at each inner step. Calling `saturate_inner` drives that loop until no fresh tuples appear. Afterwards, `LiftIntegrate` exposes the cumulative reach set at the freshest known frontier, and `latest` reads it.

In [2]:
from pydbsp.indexed_relational_operators import IndexedIncrementalReachabilityBody
from pydbsp.operator import LiftIntegrate

Edge = tuple[int, int]

eg = ZSetAddition[Edge]()
e = Evaluator[Time](
    circuit=Circuit[Time](),
    storage=DictStorage(),
    ctx=ComputeCtx(lattice=dbsp_time(2)),
    group=eg,
)

edges_1d = Input[ZSet[Edge]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
state_2d = Input[ZSet[Edge]](frontier=Antichain(dbsp_time(2))).connect(e.circuit, ())
body = IndexedIncrementalReachabilityBody[int](edge_group=eg).connect(
    e.circuit, (edges_1d, state_2d),
)
running_sum = LiftIntegrate[ZSet[Edge]](group=eg).connect(e.circuit, (body,))

e.push(edges_1d, ZSet({(1, 2): 1, (2, 3): 1, (3, 4): 1}))

for k, diff in e.saturate_inner(body, outer_tick=0, is_empty=lambda d: not d.inner):
    e.push(state_2d, diff, t=(0, k))

e.latest(running_sum).inner

{(1, 4): 1, (1, 3): 1, (2, 4): 1, (1, 2): 1, (2, 3): 1, (3, 4): 1}